# 3 — Proposed: Hybrid CNN-BiGRU-Attention (Multimodal)
**Author:** Revanth Katari

Architecture:
```
Text (768)  ─┐
Image (1280) ─┤  Project to 512  →  Stack as sequence (B, 4, 512)
Audio (768)  ─┤  →  CNN  →  BiGRU  →  Sequential Attention  →  Classify
Video (768)  ─┘
```

This model treats the four modality projections as a **4-step sequence**,
allowing CNN to extract cross-modal local patterns and BiGRU to capture
sequential dependencies between modalities.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.pardir, "src"))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from config import (
    EMBEDDINGS_DIR, MODELS_DIR, RESULTS_DIR,
    TEXT_DIM, IMAGE_DIM, AUDIO_DIM, VIDEO_DIM, HIDDEN_DIM,
    NUM_CLASSES, DROPOUT, NUM_FILTERS, BATCH_SIZE,
    LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS, PATIENCE, RANDOM_SEED,
)
from data_utils import load_embeddings, create_splits, make_dataloader
from models import HybridCNNBiGRUMultimodal
from train_utils import train_one_epoch, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
text, image, audio, video, labels = load_embeddings(EMBEDDINGS_DIR)
train_split, val_split, test_split = create_splits(
    text, image, audio, video, labels, seed=RANDOM_SEED
)
train_loader = make_dataloader(train_split, BATCH_SIZE, shuffle=True)
val_loader   = make_dataloader(val_split, BATCH_SIZE, shuffle=False)
test_loader  = make_dataloader(test_split, BATCH_SIZE, shuffle=False)
print(f"Train: {len(train_split[-1]):,}  Val: {len(val_split[-1]):,}  Test: {len(test_split[-1]):,}")

## 3.1 Model

In [ ]:
model = HybridCNNBiGRUMultimodal(
    text_dim=TEXT_DIM, image_dim=IMAGE_DIM,
    audio_dim=AUDIO_DIM, video_dim=VIDEO_DIM,
    hidden=HIDDEN_DIM, num_filters=NUM_FILTERS,
    num_classes=NUM_CLASSES, dropout=DROPOUT,
    use_cnn=True, use_attention=True,
).to(device)

print(f"Total params:     {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(model)

## 3.2 Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": [], "val_auc": []}
best_f1, wait = 0.0, 0
save_path = os.path.join(MODELS_DIR, "hybrid_cnn_bigru_attn.pt")

for epoch in range(1, NUM_EPOCHS + 1):
    tl, ta = train_one_epoch(model, train_loader, optimizer, criterion, device)
    vl, vm, _, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step(vl)

    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["train_acc"].append(ta)
    history["val_acc"].append(vm["accuracy"])
    history["val_f1"].append(vm["f1"])
    history["val_auc"].append(vm["auc"])

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
          f"TrL={tl:.4f} TrA={ta:.4f}  "
          f"VL={vl:.4f} VA={vm['accuracy']:.4f} "
          f"VF1={vm['f1']:.4f} VAUC={vm['auc']:.4f}")

    if vm["f1"] > best_f1:
        best_f1 = vm["f1"]
        torch.save(model.state_dict(), save_path)
        print(f"  -> Saved (F1={best_f1:.4f})")
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

## 3.3 Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(history["train_acc"], label="Train")
axes[1].plot(history["val_acc"], label="Val")
axes[1].set_title("Accuracy"); axes[1].legend()
axes[2].plot(history["val_f1"], label="F1", color="green")
axes[2].plot(history["val_auc"], label="AUC", color="orange")
axes[2].set_title("Val F1 / AUC"); axes[2].legend()
for ax in axes: ax.set_xlabel("Epoch")
plt.tight_layout()
plt.show()

## 3.4 Test evaluation

In [ ]:
model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
_, test_metrics, preds, probs, true_labels = evaluate(model, test_loader, criterion, device)

print("Test Metrics:")
for k, v in test_metrics.items():
    print(f"  {k:>10s}: {v:.4f}")

with open(os.path.join(RESULTS_DIR, "hybrid_cnn_bigru_attn_metrics.json"), "w") as f:
    json.dump({"test": test_metrics, "history": history}, f, indent=2)
print("Results saved.")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cm = confusion_matrix(true_labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Real", "Fake"])
disp.plot(cmap="Blues")
plt.title("Hybrid CNN-BiGRU-Attention (Multimodal)")
plt.show()

print(classification_report(true_labels, preds, target_names=["Real", "Fake"]))